In [ ]:
# 02_Boosting

In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)
from sklearn.ensemble import (
    AdaBoostClassifier, GradientBoostingClassifier, GradientBoostingRegressor
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid
from scipy.special import expit
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# -------------------------- Global Configuration --------------------------
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "axes.unicode_minus": False,
})

MODELS = [
    'DenseNet201', 'NASNetLarge', 'EfficientNetB7', 'Xception',
    'InceptionResNetV2', 'VGG19', 'ResNet50V2', 'MobileNetV3Large', 'EfficientNetV2S'
]

BASE_DIR = r'C:\Users\zhd\LungCancer'
OUTPUT_DIR = os.path.join(BASE_DIR, '03.Ensemble learning')

# -------------------------- Utility Functions --------------------------
def load_all_predictions():
    """ Load prediction csv from each base model
    Return dict containing filename, label, dataset flag and model outputs """
    dfs = {}
    for m in MODELS:
        path = os.path.join(BASE_DIR, m, 'all_predictions.csv')
        df = pd.read_csv(path)
        dfs[m] = df

    ref = dfs[MODELS[0]]
    filenames = ref['Filename'].values
    y_true = ref['True_Label'].values.astype(int)
    datasets = ref['Dataset'].values

    X_raw = np.column_stack([dfs[m]['Raw_Score'].values for m in MODELS])
    X_prob = np.column_stack([dfs[m]['Probability'].values for m in MODELS])
    X_pred = np.column_stack([dfs[m]['Predicted_Label'].values.astype(int) for m in MODELS])

    return {
        'filenames': filenames,
        'y_true': y_true,
        'datasets': datasets,
        'X_raw': X_raw,
        'X_prob': X_prob,
        'X_pred': X_pred,
        'model_names': MODELS
    }


def split_by_dataset(data, X=None):
    """ Split feature matrix and label by Dataset flag: Train / Validation / Test
    Return split data and sample index masks """
    if X is None:
        X = data['X_raw']
    y = data['y_true']
    ds = data['datasets']

    train_mask = ds == 'Train'
    val_mask = ds == 'Validation'
    test_mask = ds == 'Test'

    return {
        'X_train': X[train_mask],
        'y_train': y[train_mask],
        'X_val': X[val_mask],
        'y_val': y[val_mask],
        'X_test': X[test_mask],
        'y_test': y[test_mask],
        'train_idx': np.where(train_mask)[0],
        'val_idx': np.where(val_mask)[0],
        'test_idx': np.where(test_mask)[0]
    }


def evaluate(y_true, y_pred, y_prob):
    """ Calculate binary classification metrics: Acc, Precision, Recall, F1, AUC """
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5

    return {
        'Accuracy': round(acc, 6),
        'Precision': round(prec, 6),
        'Recall': round(rec, 6),
        'F1': round(f1, 6),
        'AUC': round(auc, 6)
    }


def find_optimal_threshold(y_true, prob):
    """ Search threshold by maximize F1 score, using ROC curve thresholds """
    fpr, tpr, thresholds = roc_curve(y_true, prob)
    best_f1 = 0
    best_thresh = 0.5
    for thresh in thresholds:
        if thresh <= 0 or thresh >= 1:
            continue
        pred = (prob >= thresh).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    return best_thresh, best_f1


def save_results_table(results, category_name, output_dir):
    """ Save atomic performance table for all ensemble algorithms
    Sort by validation F1 descending """
    rows = []
    for algo_name, res in results.items():
        row = {'Algorithm': algo_name, 'Category': category_name}
        for split in ['train', 'val', 'test']:
            metrics = res[f'{split}_metrics']
            for k, v in metrics.items():
                row[f'{split}_{k}'] = v
        rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sort_values('val_F1', ascending=False).reset_index(drop=True)
    path = os.path.join(output_dir, f'{category_name}_results.csv')
    df.to_csv(path, index=False)
    return df


def plot_roc_curves(results, category_name, output_dir, top_n=10):
    """ Plot ROC curves based on validation set """
    fig, ax = plt.subplots(figsize=(16, 12))
    sorted_res = sorted(results.items(), key=lambda x: x[1]['val_metrics']['F1'], reverse=True)
    cmap = plt.cm.tab20

    for i, (name, res) in enumerate(sorted_res[:top_n]):
        fpr, tpr, _ = roc_curve(res['y_val'], res['val_prob'])
        auc = res['val_metrics']['AUC']
        ax.plot(fpr, tpr, color=cmap(i % 20), linewidth=2.5, label=f'{name} (AUC={auc:.4f})')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5)
    ax.set_xlabel('False Positive Rate', fontsize=40)
    ax.set_ylabel('True Positive Rate', fontsize=40)
    ax.set_title(f'{category_name} - ROC Curves (Validation Set)', fontsize=44)
    ax.tick_params(axis='both', labelsize=36)
    ax.legend(fontsize=20, loc='lower right', framealpha=0.9)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    plt.tight_layout()

    path = os.path.join(output_dir, f'{category_name}_ROC_curves.pdf')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()


def plot_metric_comparison(results, category_name, output_dir):
    """ Bar chart comparison between validation and test set metrics """
    sorted_res = sorted(results.items(), key=lambda x: x[1]['val_metrics']['F1'], reverse=True)
    names = [r[0] for r in sorted_res]
    metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']

    fig, axes = plt.subplots(1, 5, figsize=(40, 12))
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

    for idx, metric in enumerate(metrics_list):
        ax = axes[idx]
        val_vals = [r[1]['val_metrics'][metric] for r in sorted_res]
        test_vals = [r[1]['test_metrics'][metric] for r in sorted_res]

        x = np.arange(len(names))
        width = 0.35
        ax.bar(x - width/2, val_vals, width, label='Validation', color=colors[idx], alpha=0.6)
        ax.bar(x + width/2, test_vals, width, label='Test', color=colors[idx], alpha=1.0)

        ax.set_ylabel(metric, fontsize=36)
        ax.set_title(metric, fontsize=40)
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=45, ha='right', fontsize=18)
        ax.tick_params(axis='y', labelsize=32)
        ax.legend(fontsize=24)
        ax.set_ylim([0.5, 1.02])

    plt.suptitle(f'{category_name} - Metric Comparison', fontsize=48, y=1.02)
    plt.tight_layout()

    path = os.path.join(output_dir, f'{category_name}_metric_comparison.pdf')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()


def save_sample_predictions(algo_name, data, pred_all, prob_all, raw_score_all, output_dir):
    """ Save per‑sample prediction for single algorithm
    Columns: Filename, True_Label, Predicted_Label, Raw_Score, Probability, Correct, Dataset """
    df = pd.DataFrame({
        'Filename': data['filenames'],
        'True_Label': data['y_true'],
        'Predicted_Label': pred_all.astype(int),
        'Raw_Score': raw_score_all,
        'Probability': prob_all,
        'Correct': pred_all.astype(int) == data['y_true'],
        'Dataset': data['datasets']
    })
    path = os.path.join(output_dir, f'{algo_name}_predictions.csv')
    df.to_csv(path, index=False)
    return df


# -------------------------- Custom Boosting Class Definitions --------------------------
class LogitBoost:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=1):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.models = []

    def fit(self, X, y):
        self.models = []
        n_samples = len(X)
        F = np.zeros(n_samples)
        y_signed = 2 * y - 1

        for _ in range(self.n_estimators):
            p = expit(F)
            p = np.clip(p, 1e-6, 1 - 1e-6)
            z = y_signed / (p * (1 - p))
            weights = p * (1 - p)
            weights = np.clip(weights, 1e-6, None)
            weights = weights / (weights.sum() + 1e-10)

            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=42)
            tree.fit(X, z, sample_weight=weights)
            pred = tree.predict(X)

            F += self.learning_rate * 0.5 * pred
            self.models.append(tree)
        return self

    def predict_proba(self, X):
        F = np.zeros(len(X))
        for tree in self.models:
            F += self.learning_rate * 0.5 * tree.predict(X)
        prob1 = expit(F)
        return np.column_stack([1 - prob1, prob1])


class SAMMEClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=1):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.models = []
        self.alphas = []
        self.classes = None

    def fit(self, X, y):
        self.models = []
        self.alphas = []
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        n_samples = len(X)
        sample_weights = np.ones(n_samples) / n_samples

        for _ in range(self.n_estimators):
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=42)
            tree.fit(X, y, sample_weight=sample_weights)
            pred = tree.predict(X)
            incorrect = (pred != y).astype(float)
            error = np.sum(sample_weights * incorrect)

            if error < 1e-10:
                alpha = 1.0
            elif error >= 1 - 1.0 / n_classes:
                alpha = 0.01
            else:
                alpha = self.learning_rate * np.log((1 - error) / (error + 1e-10)) + np.log(n_classes - 1)

            self.models.append(tree)
            self.alphas.append(alpha)
            sample_weights = sample_weights * np.exp(alpha * incorrect)
            sample_weights = sample_weights / (sample_weights.sum() + 1e-10)
        return self

    def predict_proba(self, X):
        n_classes = len(self.classes)
        probas = np.zeros((len(X), n_classes))
        for tree, alpha in zip(self.models, self.alphas):
            pred = tree.predict(X)
            for i, c in enumerate(self.classes):
                probas[:, i] += alpha * (pred == c).astype(float)
        probas = np.exp(probas)
        probas = probas / (probas.sum(axis=1, keepdims=True) + 1e-10)
        return probas


class BrownBoost:
    def __init__(self, n_estimators=100, c=1.0, max_depth=1):
        self.n_estimators = n_estimators
        self.c = c
        self.max_depth = max_depth
        self.models = []
        self.alphas = []

    def fit(self, X, y):
        self.models = []
        self.alphas = []
        n_samples = len(X)
        y_signed = 2 * y - 1
        r = np.zeros(n_samples)
        remaining_time = self.c

        for _ in range(self.n_estimators):
            if remaining_time <= 1e-6:
                break
            weights = np.exp(-r ** 2 / self.c)
            weights = weights / (weights.sum() + 1e-10)

            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=42)
            tree.fit(X, y, sample_weight=weights)
            pred_signed = 2 * tree.predict(X) - 1
            error = np.sum(weights * (pred_signed != y_signed))

            if error <= 0 or error >= 0.5:
                alpha = 0.5 * remaining_time if error <= 0 else 0
            else:
                gamma = (1 - error) / error
                alpha = 0.5 * np.log(gamma)
                alpha = min(alpha, remaining_time)

            r += alpha * y_signed * pred_signed
            remaining_time -= alpha
            self.models.append(tree)
            self.alphas.append(alpha)
        return self

    def predict_proba(self, X):
        F = np.zeros(len(X))
        for tree, alpha in zip(self.models, self.alphas):
            pred_signed = 2 * tree.predict(X) - 1
            F += alpha * pred_signed
        prob1 = expit(F)
        return np.column_stack([1 - prob1, prob1])


class LPBoost:
    def __init__(self, n_estimators=100, max_depth=1, C=1.0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.C = C
        self.models = []
        self.weights = []

    def fit(self, X, y):
        self.models = []
        self.weights = []
        n_samples = len(X)
        y_signed = 2 * y - 1
        sample_weights = np.ones(n_samples) / n_samples

        for _ in range(self.n_estimators):
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=42)
            tree.fit(X, y, sample_weight=sample_weights)
            pred_signed = 2 * tree.predict(X) - 1
            margin = y_signed * pred_signed
            incorrect = (margin < 0).astype(float)
            error = np.sum(sample_weights * incorrect)

            if error < 1e-10:
                alpha = self.C
            elif error >= 0.5:
                alpha = 0.01
            else:
                alpha = 0.5 * np.log((1 - error) / (error + 1e-10))

            self.models.append(tree)
            self.weights.append(alpha)
            sample_weights = sample_weights * np.exp(-alpha * margin)
            sample_weights = np.clip(sample_weights, 0, self.C / n_samples)
            sample_weights = sample_weights / (sample_weights.sum() + 1e-10)

        self.weights = np.array(self.weights)
        if self.weights.sum() > 0:
            self.weights = self.weights / self.weights.sum()
        return self

    def predict_proba(self, X):
        F = np.zeros(len(X))
        for tree, weight in zip(self.models, self.weights):
            pred_signed = 2 * tree.predict(X) - 1
            F += weight * pred_signed
        prob1 = expit(F)
        return np.column_stack([1 - prob1, prob1])


class GBRTClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, subsample=1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.subsample = subsample
        self.model = None

    def fit(self, X, y):
        self.model = GradientBoostingRegressor(
            n_estimators=self.n_estimators,
            learning_rate=self.learning_rate,
            max_depth=self.max_depth,
            subsample=self.subsample,
            random_state=42
        )
        y_signed = 2 * y - 1
        self.model.fit(X, y_signed)
        return self

    def predict_proba(self, X):
        raw = self.model.predict(X)
        prob1 = expit(raw)
        return np.column_stack([1 - prob1, prob1])


# -------------------------- Main Experiment Pipeline --------------------------
CATEGORY = 'Boosting'
OUT_DIR = os.path.join(OUTPUT_DIR, '02_Boosting')
os.makedirs(OUT_DIR, exist_ok=True)

data = load_all_predictions()
splits = split_by_dataset(data)

X_train, y_train = splits['X_train'], splits['y_train']
X_val, y_val = splits['X_val'], splits['y_val']
X_test, y_test = splits['X_test'], splits['y_test']
train_idx, val_idx, test_idx = splits['train_idx'], splits['val_idx'], splits['test_idx']

results = {}


def run_boosting_algo(name, model_builder, param_grid):
    """
    Hyper‑parameter search on TRAIN set, threshold optimize on TRAIN set
    Validation set for independent evaluation; Test set reserved for weight combination
    Save full sample predictions and store metrics into results dict
    """
    best_f1 = -1
    best_params = None
    best_model = None

    # Hyper‑parameter grid search using Train set
    for params in ParameterGrid(param_grid):
        model = model_builder(params)
        model.fit(X_train, y_train)

        if hasattr(model, 'predict_proba'):
            prob_eval = model.predict_proba(X_train)[:, 1]
        else:
            df_eval = model.decision_function(X_train)
            prob_eval = (df_eval - df_eval.min()) / (df_eval.max() - df_eval.min() + 1e-10)

        thresh, _ = find_optimal_threshold(y_train, prob_eval)
        pred_eval = (prob_eval >= thresh).astype(int)
        f1 = f1_score(y_train, pred_eval, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_params = params
            best_model = model

    model = best_model

    # Obtain raw scores and probability for all splits
    if hasattr(model, 'predict_proba'):
        prob_train = model.predict_proba(X_train)[:, 1]
        prob_val = model.predict_proba(X_val)[:, 1]
        prob_test = model.predict_proba(X_test)[:, 1]
        raw_train, raw_val, raw_test = prob_train.copy(), prob_val.copy(), prob_test.copy()
    else:
        df_train = model.decision_function(X_train)
        df_val = model.decision_function(X_val)
        df_test = model.decision_function(X_test)
        all_df = np.concatenate([df_train, df_val, df_test])
        mn, mx = all_df.min(), all_df.max()
        prob_train = (df_train - mn) / (mx - mn + 1e-10)
        prob_val = (df_val - mn) / (mx - mn + 1e-10)
        prob_test = (df_test - mn) / (mx - mn + 1e-10)
        raw_train, raw_val, raw_test = df_train.copy(), df_val.copy(), df_test.copy()

    # Optimize threshold on Train set only
    thresh, _ = find_optimal_threshold(y_train, prob_train)
    pred_train = (prob_train >= thresh).astype(int)
    pred_val = (prob_val >= thresh).astype(int)
    pred_test = (prob_test >= thresh).astype(int)

    # Assemble full‑length arrays for all samples
    n_samples = len(data['filenames'])
    pred_all = np.zeros(n_samples, dtype=int)
    prob_all = np.zeros(n_samples, dtype=float)
    raw_all = np.zeros(n_samples, dtype=float)

    pred_all[train_idx] = pred_train
    pred_all[val_idx] = pred_val
    pred_all[test_idx] = pred_test
    prob_all[train_idx] = prob_train
    prob_all[val_idx] = prob_val
    prob_all[test_idx] = prob_test
    raw_all[train_idx] = raw_train
    raw_all[val_idx] = raw_val
    raw_all[test_idx] = raw_test

    save_sample_predictions(name, data, pred_all, prob_all, raw_all, OUT_DIR)

    results[name] = {
        'best_params': best_params,
        'threshold': thresh,
        'train_metrics': evaluate(y_train, pred_train, prob_train),
        'val_metrics': evaluate(y_val, pred_val, prob_val),
        'test_metrics': evaluate(y_test, pred_test, prob_test),
        'y_val': y_val,
        'val_prob': prob_val,
        'y_test': y_test,
        'test_prob': prob_test
    }

    print(f'{name}: Val F1={results[name]["val_metrics"]["F1"]:.4f}, '
          f'Test F1={results[name]["test_metrics"]["F1"]:.4f}, '
          f'Val AUC={results[name]["val_metrics"]["AUC"]:.4f}')


# -------------------------- Launch Each Ensemble Algorithm --------------------------
run_boosting_algo(
    'AdaBoost',
    lambda p: AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=p['max_depth']),
        n_estimators=p['n_estimators'],
        learning_rate=p['learning_rate'],
        random_state=42
    ),
    {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.5],
        'max_depth': [1, 2, 3]
    }
)

run_boosting_algo(
    'LogitBoost',
    lambda p: LogitBoost(
        n_estimators=p['n_estimators'],
        learning_rate=p['learning_rate'],
        max_depth=p['max_depth']
    ),
    {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.5],
        'max_depth': [1, 2, 3]
    }
)

run_boosting_algo(
    'GBDT',
    lambda p: GradientBoostingClassifier(
        n_estimators=p['n_estimators'],
        learning_rate=p['learning_rate'],
        max_depth=p['max_depth'],
        min_samples_split=p['min_samples_split'],
        subsample=p['subsample'],
        random_state=42
    ),
    {
        'n_estimators': [100, 200, 500],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [2, 3, 5],
        'min_samples_split': [2, 5, 10],
        'subsample': [0.7, 0.85, 1.0]
    }
)

run_boosting_algo(
    'XGBoost',
    lambda p: xgb.XGBClassifier(
        n_estimators=p['n_estimators'],
        learning_rate=p['learning_rate'],
        max_depth=p['max_depth'],
        subsample=p['subsample'],
        colsample_bytree=p['colsample_bytree'],
        reg_alpha=p['reg_alpha'],
        reg_lambda=p['reg_lambda'],
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss',
        n_jobs=-1
    ),
    {
        'n_estimators': [100, 200, 500],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [2, 3, 5],
        'subsample': [0.7, 0.85, 1.0],
        'colsample_bytree': [0.7, 0.85, 1.0],
        'reg_alpha': [0, 0.1, 1.0],
        'reg_lambda': [0.1, 1.0, 10.0]
    }
)

run_boosting_algo(
    'LightGBM',
    lambda p: lgb.LGBMClassifier(
        n_estimators=p['n_estimators'],
        learning_rate=p['learning_rate'],
        num_leaves=p['num_leaves'],
        subsample=p['subsample'],
        colsample_bytree=p['colsample_bytree'],
        reg_alpha=p['reg_alpha'],
        reg_lambda=p['reg_lambda'],
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    {
        'n_estimators': [100, 200, 500],
        'learning_rate': [0.01, 0.05, 0.1],
        'num_leaves': [15, 31, 63],
        'subsample': [0.7, 0.85, 1.0],
        'colsample_bytree': [0.7, 0.85, 1.0],
        'reg_alpha': [0, 0.1, 1.0],
        'reg_lambda': [0.1, 1.0, 10.0]
    }
)

run_boosting_algo(
    'CatBoost',
    lambda p: cb.CatBoostClassifier(
        iterations=p['iterations'],
        learning_rate=p['learning_rate'],
        depth=p['depth'],
        l2_leaf_reg=p['l2_leaf_reg'],
        subsample=p['subsample'],
        random_state=42,
        verbose=0
    ),
    {
        'iterations': [100, 200, 500],
        'learning_rate': [0.01, 0.05, 0.1],
        'depth': [2, 4, 6],
        'l2_leaf_reg': [1, 3, 10],
        'subsample': [0.7, 0.85, 1.0]
    }
)

run_boosting_algo(
    'SAMME',
    lambda p: SAMMEClassifier(
        n_estimators=p['n_estimators'],
        learning_rate=p['learning_rate'],
        max_depth=p['max_depth']
    ),
    {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.5],
        'max_depth': [1, 2, 3]
    }
)

run_boosting_algo(
    'BrownBoost',
    lambda p: BrownBoost(
        n_estimators=p['n_estimators'],
        c=p['c'],
        max_depth=p['max_depth']
    ),
    {
        'n_estimators': [50, 100, 200],
        'c': [0.5, 1.0, 2.0],
        'max_depth': [1, 2, 3]
    }
)

run_boosting_algo(
    'LPBoost',
    lambda p: LPBoost(
        n_estimators=p['n_estimators'],
        max_depth=p['max_depth'],
        C=p['C']
    ),
    {
        'n_estimators': [50, 100, 200],
        'max_depth': [1, 2, 3],
        'C': [0.5, 1.0, 2.0]
    }
)

run_boosting_algo(
    'GBRT',
    lambda p: GBRTClassifier(
        n_estimators=p['n_estimators'],
        learning_rate=p['learning_rate'],
        max_depth=p['max_depth'],
        subsample=p['subsample']
    ),
    {
        'n_estimators': [100, 200, 500],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [2, 3, 5],
        'subsample': [0.7, 0.85, 1.0]
    }
)

# -------------------------- Export Basic Results --------------------------
df_results = save_results_table(results, CATEGORY, OUT_DIR)
plot_roc_curves(results, CATEGORY, OUT_DIR)
plot_metric_comparison(results, CATEGORY, OUT_DIR)

print(f'\n{CATEGORY} complete. Results saved to {OUT_DIR}')
print('\nValidation Set Performance (Independent Evaluation):')
print(df_results[['Algorithm', 'val_Accuracy', 'val_Precision', 'val_Recall', 'val_F1', 'val_AUC']].to_string(index=False))


AdaBoost: Val F1=0.6732, Test F1=0.8840, Val AUC=0.5145
LogitBoost: Val F1=0.7823, Test F1=0.8885, Val AUC=0.7220
GBDT: Val F1=0.8126, Test F1=0.8677, Val AUC=0.6334
XGBoost: Val F1=0.9387, Test F1=0.8225, Val AUC=0.9684
LightGBM: Val F1=0.7735, Test F1=0.8641, Val AUC=0.7800
CatBoost: Val F1=0.7967, Test F1=0.8677, Val AUC=0.6710
SAMME: Val F1=0.6667, Test F1=0.8773, Val AUC=0.7220
BrownBoost: Val F1=0.7823, Test F1=0.8885, Val AUC=0.7220
LPBoost: Val F1=0.7823, Test F1=0.8885, Val AUC=0.7220
GBRT: Val F1=0.8129, Test F1=0.8667, Val AUC=0.6256

Boosting complete. Results saved to C:\Users\zhd\LungCancer\03.Ensemble learning\02_Boosting

Validation Set Performance (Independent Evaluation):
 Algorithm  val_Accuracy  val_Precision  val_Recall   val_F1  val_AUC
   XGBoost        0.9375       0.921078       0.957 0.938695 0.968367
      GBRT        0.7715       0.688150       0.993 0.812935 0.625555
      GBDT        0.7710       0.687673       0.993 0.812602 0.633416
  CatBoost        0.7

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
from sklearn.calibration import calibration_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "axes.unicode_minus": False,
    "axes.titlesize": 40,
    "axes.labelsize": 40,
    "xtick.labelsize": 40,
    "ytick.labelsize": 40,
    "legend.fontsize": 40,
})

BASE_DIR = r'C:\Users\zhd\LungCancer'
OUTPUT_DIR = os.path.join(BASE_DIR, '03.Ensemble learning', '02_Boosting')

COLOR_TRAIN = "#1f77b4"
COLOR_VAL = "#ff7f0e"
COLOR_TEST = "#2ca02c"


def safe_get(d, key):
    if key not in d:
        raise KeyError(f"Missing key '{key}'. Available keys: {list(d.keys())}")
    return d[key]


def debug_print_results_keys(results):
    print("\n==== DEBUG: keys in first algorithm result ====")
    first_key = list(results.keys())[0]
    print(f"Algorithm: {first_key}")
    print(f"Keys: {list(results[first_key].keys())}\n")


def bootstrap_metric_ci(y_true, y_pred, y_prob, n_boot=1000):
    rng = np.random.RandomState(42)
    metrics_boot = {"Accuracy": [], "Precision": [], "Recall": [], "F1": [], "AUC": []}
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        yt = y_true[idx]
        yp = y_pred[idx]
        ypr = y_prob[idx]
        try:
            acc = accuracy_score(yt, yp)
            prec = precision_score(yt, yp, zero_division=0)
            rec = recall_score(yt, yp, zero_division=0)
            f1 = f1_score(yt, yp, zero_division=0)
            auc = roc_auc_score(yt, ypr)
        except ValueError:
            continue
        metrics_boot["Accuracy"].append(acc)
        metrics_boot["Precision"].append(prec)
        metrics_boot["Recall"].append(rec)
        metrics_boot["F1"].append(f1)
        metrics_boot["AUC"].append(auc)

    out = {}
    for k, v in metrics_boot.items():
        arr = np.array(v)
        out[f"{k}_mean"] = np.mean(arr)
        out[f"{k}_ci_low"] = np.percentile(arr, 2.5)
        out[f"{k}_ci_high"] = np.percentile(arr, 97.5)
    return out


def bootstrap_auc_ci(y_true, y_score, n_boot=1000):
    rng = np.random.RandomState(42)
    aucs = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        try:
            auc = roc_auc_score(y_true[idx], y_score[idx])
            aucs.append(auc)
        except ValueError:
            continue
    aucs = np.array(aucs)
    mean_auc = roc_auc_score(y_true, y_score)
    ci_low = np.percentile(aucs, 2.5)
    ci_high = np.percentile(aucs, 97.5)
    return mean_auc, ci_low, ci_high


def save_fig(fig, base_name, out_dir, dpi=300):
    fig.savefig(os.path.join(out_dir, f"{base_name}.pdf"), dpi=dpi, bbox_inches="tight")
    fig.savefig(os.path.join(out_dir, f"{base_name}.png"), dpi=dpi, bbox_inches="tight")
    plt.close(fig)


def plot_metric_comparison(results, category_name, output_dir):
    sorted_res = sorted(results.items(), key=lambda x: safe_get(x[1]['val_metrics'], 'F1'), reverse=True)
    names = [r[0] for r in sorted_res]
    metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']

    fig, axes = plt.subplots(1, 5, figsize=(54, 20))
    colors = [COLOR_TRAIN, COLOR_TEST, COLOR_VAL]
    group_names = ["Train", "Test", "Validation"]
    width = 0.25
    x = np.arange(len(names))

    for idx_m, metric in enumerate(metrics_list):
        ax = axes[idx_m]
        train_vals = [safe_get(r[1]['train_metrics'], metric) for r in sorted_res]
        test_vals = [safe_get(r[1]['test_metrics'], metric) for r in sorted_res]
        val_vals = [safe_get(r[1]['val_metrics'], metric) for r in sorted_res]

        ax.bar(x - width, train_vals, width, label=group_names[0], color=colors[0], alpha=0.75)
        ax.bar(x, test_vals, width, label=group_names[1], color=colors[1], alpha=0.75)
        ax.bar(x + width, val_vals, width, label=group_names[2], color=colors[2], alpha=0.75)

        ax.set_ylabel(metric, fontsize=40)
        ax.set_title(metric, fontsize=40)
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=40, ha="right", fontsize=40)
        ax.tick_params(axis='y', labelsize=40)
        ax.set_ylim([0.5, 1.02])

    fig.legend(labels=group_names, loc="lower center", ncol=3, fontsize=40, bbox_to_anchor=(0.5, 0.01))
    plt.suptitle(f"{category_name} Metric Comparison", fontsize=40, y=0.98)
    plt.tight_layout(rect=[0, 0.10, 1, 0.96])
    save_fig(fig, f"{category_name}_metric_comparison", output_dir)


def plot_roc_curves(results, category_name, output_dir):
    algo_list = list(results.keys())
    n_alg = len(algo_list)
    n_col = 3
    n_row = int(np.ceil(n_alg / n_col))
    sub_w, sub_h = 12, 12

    fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * sub_w, n_row * sub_h))
    axes = axes.flatten()

    dataset_info = [
        ("Train", COLOR_TRAIN),
        ("Test", COLOR_TEST),
        ("Validation", COLOR_VAL),
    ]

    for idx, algo in enumerate(algo_list):
        ax = axes[idx]
        res = results[algo]
        for ds_name, color in dataset_info:
            if ds_name == "Train":
                train_m = res["train_metrics"]
                auc_val = train_m["AUC"]
                ax.plot([], [], color=color, lw=3, label=f"{ds_name} AUC={auc_val:.3f} (No raw data)")
            elif ds_name == "Test":
                y = safe_get(res, "y_test")
                prob = safe_get(res, "test_prob")
                mean_auc, ci_low, ci_high = bootstrap_auc_ci(y, prob)
                fpr, tpr, _ = roc_curve(y, prob)
                label_str = f"{ds_name} AUC={mean_auc:.3f} (95%CI:{ci_low:.3f}-{ci_high:.3f})"
                ax.plot(fpr, tpr, color=color, lw=3, label=label_str)
            elif ds_name == "Validation":
                y = safe_get(res, "y_val")
                prob = safe_get(res, "val_prob")
                mean_auc, ci_low, ci_high = bootstrap_auc_ci(y, prob)
                fpr, tpr, _ = roc_curve(y, prob)
                label_str = f"{ds_name} AUC={mean_auc:.3f} (95%CI:{ci_low:.3f}-{ci_high:.3f})"
                ax.plot(fpr, tpr, color=color, lw=3, label=label_str)

        ax.plot([0, 1], [0, 1], "k--", lw=3, alpha=0.6)
        ax.set_aspect("equal")
        ax.set_xlabel("False Positive Rate", fontsize=40)
        ax.set_ylabel("True Positive Rate", fontsize=40)
        ax.set_title(f"{algo}", fontsize=40)
        ax.tick_params(axis="both", labelsize=40)
        ax.legend(loc="lower right", frameon=True, fontsize=40)
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)

    for idx in range(len(algo_list), len(axes)):
        axes[idx].axis("off")

    plt.suptitle(f"{category_name} ROC Curves (Train / Test / Validation)", fontsize=40, y=1.03)
    plt.tight_layout()
    save_fig(fig, f"{category_name}_ROC_curves", output_dir)


def plot_calibration_all(results, category_name, output_dir):
    algo_list = list(results.keys())
    n_alg = len(algo_list)
    n_col = 3
    n_row = int(np.ceil(n_alg / n_col))
    sub_w, sub_h = 16, 12

    fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * sub_w, n_row * sub_h))
    axes = axes.flatten()

    ds_spec = [
        ("Train", COLOR_TRAIN),
        ("Test", COLOR_TEST),
        ("Val", COLOR_VAL),
    ]

    for i, algo in enumerate(algo_list):
        ax = axes[i]
        res = results[algo]
        for ds_name, c in ds_spec:
            if ds_name == "Train":
                ax.plot([], [], color=c, lw=3, marker="o", label=f"{ds_name}: No raw probability")
            elif ds_name == "Test":
                y_true = safe_get(res, "y_test")
                prob = safe_get(res, "test_prob")
                prob_clipped = np.clip(prob, 0.0, 1.0)
                frac_pos, mean_pred = calibration_curve(y_true, prob_clipped, n_bins=10)
                ax.plot(mean_pred, frac_pos, marker="o", lw=3, label=ds_name, color=c)
            elif ds_name == "Val":
                y_true = safe_get(res, "y_val")
                prob = safe_get(res, "val_prob")
                prob_clipped = np.clip(prob, 0.0, 1.0)
                frac_pos, mean_pred = calibration_curve(y_true, prob_clipped, n_bins=10)
                ax.plot(mean_pred, frac_pos, marker="o", lw=3, label=ds_name, color=c)

        ax.plot([0, 1], [0, 1], "k--", lw=3)
        ax.set_xlabel("Mean predicted probability", fontsize=40)
        ax.set_ylabel("Fraction of positives", fontsize=40)
        ax.set_title(f"{algo} Calibration Curve", fontsize=40)
        ax.legend(fontsize=40)
        ax.tick_params(labelsize=40)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

    for j in range(len(algo_list), len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"{category_name} Calibration Curves (Train / Test / Validation)", fontsize=40, y=1.03)
    plt.tight_layout()
    save_fig(fig, f"{category_name}_calibration", output_dir)


def plot_dca_all(results, category_name, output_dir):
    algo_list = list(results.keys())
    n_alg = len(algo_list)
    n_col = 3
    n_row = int(np.ceil(n_alg / n_col))
    sub_w, sub_h = 12, 12

    fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * sub_w, n_row * sub_h))
    axes = axes.flatten()
    threshold_range = np.linspace(0.01, 0.99, 100)

    ds_spec = [
        ("Train", COLOR_TRAIN),
        ("Test", COLOR_TEST),
        ("Val", COLOR_VAL),
    ]

    for idx, algo in enumerate(algo_list):
        ax = axes[idx]
        res = results[algo]
        for ds_name, color in ds_spec:
            if ds_name == "Train":
                ax.plot([], [], color=color, lw=3, label=f"{ds_name}: No raw data")
            elif ds_name == "Test":
                y = safe_get(res, "y_test")
                prob = safe_get(res, "test_prob")
                net_benefit = []
                for t in threshold_range:
                    pred = (prob >= t).astype(int)
                    tp = np.sum((pred == 1) & (y == 1))
                    fp = np.sum((pred == 1) & (y == 0))
                    n = len(y)
                    nb = (tp / n) - (fp / n) * (t / (1 - t)) if not (t == 0 or t == 1) else 0
                    net_benefit.append(nb)
                ax.plot(threshold_range, net_benefit, lw=3, label=ds_name, color=color)
            elif ds_name == "Val":
                y = safe_get(res, "y_val")
                prob = safe_get(res, "val_prob")
                net_benefit = []
                for t in threshold_range:
                    pred = (prob >= t).astype(int)
                    tp = np.sum((pred == 1) & (y == 1))
                    fp = np.sum((pred == 1) & (y == 0))
                    n = len(y)
                    nb = (tp / n) - (fp / n) * (t / (1 - t)) if not (t == 0 or t == 1) else 0
                    net_benefit.append(nb)
                ax.plot(threshold_range, net_benefit, lw=3, label=ds_name, color=color)

        ax.axhline(y=0, color="#777777", linestyle="--", label="Treat none", lw=3)
        ax.set_xlabel("Threshold probability", fontsize=40)
        ax.set_ylabel("Net benefit", fontsize=40)
        ax.set_title(f"{algo} DCA", fontsize=40)
        ax.legend(fontsize=40)
        ax.tick_params(labelsize=40)

    for j in range(len(algo_list), len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"{category_name} Decision Curve Analysis (DCA)", fontsize=40, y=1.03)
    plt.tight_layout()
    save_fig(fig, f"{category_name}_DCA", output_dir)


def plot_nomogram_density_all(results, category_name, output_dir):
    algo_list = list(results.keys())
    n_alg = len(algo_list)
    n_col = 3
    n_row = int(np.ceil(n_alg / n_col))
    sub_w, sub_h = 12, 12

    fig = plt.figure(figsize=(n_col * sub_w, n_row * sub_h))
    gs = GridSpec(n_row, n_col, figure=fig)

    for idx, algo in enumerate(algo_list):
        row_pos = idx // n_col
        col_pos = idx % n_col
        res = results[algo]
        ax_main = fig.add_subplot(gs[row_pos, col_pos])

        prob_all = np.concatenate([safe_get(res, "test_prob"), safe_get(res, "val_prob")])
        y_all = np.concatenate([safe_get(res, "y_test"), safe_get(res, "y_val")])

        ax_main.hist(prob_all, bins=30, density=True, alpha=0.25, color="#aaaaaa", zorder=0)
        jitter = np.random.normal(0, 0.04, size=len(prob_all))

        for label_val, c, marker in zip([0, 1], ["#1f77b4", "#d62728"], ["o", "^"]):
            mask = y_all == label_val
            ax_main.scatter(prob_all[mask], label_val + jitter[mask], s=60, alpha=0.65, color=c, marker=marker)

        ax_main.set_xlim(0, 1)
        ax_main.set_ylim(-0.3, 1.3)
        ax_main.set_yticks([0, 1])
        ax_main.set_yticklabels(["Negative", "Positive"], fontsize=40)
        ax_main.set_xlabel("Predicted Probability", fontsize=40)
        ax_main.set_title(f"{algo} Nomogram‑Density", fontsize=40)
        ax_main.tick_params(labelsize=40)

    plt.suptitle(f"{category_name} Nomogram with Sample Density", fontsize=40, y=1.03)
    plt.tight_layout()
    save_fig(fig, f"{category_name}_nomogram_density", output_dir)


def merge_all_predictions(results, output_dir):
    collect = []
    for algo_name in results.keys():
        csv_path = os.path.join(output_dir, f"{algo_name}_predictions.csv")
        df = pd.read_csv(csv_path)
        df = df.copy()
        df.loc[df["Raw_Score"].isna(), "Raw_Score"] = "--"
        df = df.rename(columns={
            "Predicted_Label": f"{algo_name}_PredLabel",
            "Raw_Score": f"{algo_name}_RawScore",
            "Probability": f"{algo_name}_Prob",
        })
        keep_cols = ["Filename", "True_Label", "Dataset",
                     f"{algo_name}_PredLabel", f"{algo_name}_RawScore", f"{algo_name}_Prob"]
        collect.append(df[keep_cols])

    base_df = collect[0][["Filename", "True_Label", "Dataset"]].copy()
    for df_sub in collect:
        merge_cols = ["Filename", "True_Label", "Dataset"]
        base_df = pd.merge(base_df, df_sub, on=merge_cols, how="left")

    out_file = os.path.join(output_dir, "all_predictions.csv")
    base_df.to_csv(out_file, index=False)
    print(f"Merged prediction saved: {out_file}")
    return base_df


def export_all_evaluation_with_ci(results, output_dir, filename="all_evaluation_metrics.csv"):
    metric_list = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
    rows = []

    for algo_name, res in results.items():
        for ds_name in ["Train", "Val", "Test"]:
            row = {
                "Category": "Boosting",
                "Algorithm": algo_name,
                "Dataset": ds_name
            }
            if ds_name == "Train":
                m = res["train_metrics"]
                for met in metric_list:
                    row[met] = m[met]
                    row[f"{met}_CI95_low"] = np.nan
                    row[f"{met}_CI95_high"] = np.nan
            elif ds_name == "Val":
                y_true = safe_get(res, "y_val")
                thresh = safe_get(res, "threshold")
                y_pred = (safe_get(res, "val_prob") >= thresh).astype(int)
                y_prob = safe_get(res, "val_prob")
                ci_dict = bootstrap_metric_ci(y_true, y_pred, y_prob, n_boot=1000)
                for met in metric_list:
                    row[met] = round(ci_dict[f"{met}_mean"], 6)
                    row[f"{met}_CI95_low"] = round(ci_dict[f"{met}_ci_low"], 6)
                    row[f"{met}_CI95_high"] = round(ci_dict[f"{met}_ci_high"], 6)
            elif ds_name == "Test":
                y_true = safe_get(res, "y_test")
                thresh = safe_get(res, "threshold")
                y_pred = (safe_get(res, "test_prob") >= thresh).astype(int)
                y_prob = safe_get(res, "test_prob")
                ci_dict = bootstrap_metric_ci(y_true, y_pred, y_prob, n_boot=1000)
                for met in metric_list:
                    row[met] = round(ci_dict[f"{met}_mean"], 6)
                    row[f"{met}_CI95_low"] = round(ci_dict[f"{met}_ci_low"], 6)
                    row[f"{met}_CI95_high"] = round(ci_dict[f"{met}_ci_high"], 6)
            rows.append(row)

    df_out = pd.DataFrame(rows)
    csv_path = os.path.join(output_dir, filename)
    df_out.to_csv(csv_path, index=False)
    print(f"Evaluation table with bootstrap CI saved: {csv_path}")
    return df_out


# ========== Execute ========== 
CATEGORY = "Boosting"
os.makedirs(OUTPUT_DIR, exist_ok=True)

debug_print_results_keys(results)
plot_metric_comparison(results, CATEGORY, OUTPUT_DIR)
plot_roc_curves(results, CATEGORY, OUTPUT_DIR)
plot_calibration_all(results, CATEGORY, OUTPUT_DIR)
plot_dca_all(results, CATEGORY, OUTPUT_DIR)
plot_nomogram_density_all(results, CATEGORY, OUTPUT_DIR)
merge_all_predictions(results, OUTPUT_DIR)
export_all_evaluation_with_ci(results, OUTPUT_DIR, filename="all_evaluation_metrics.csv")

print(f"All figures & tables saved to {OUTPUT_DIR}")



==== DEBUG: keys in first algorithm result ====
Algorithm: AdaBoost
Keys: ['best_params', 'threshold', 'train_metrics', 'val_metrics', 'test_metrics', 'y_val', 'val_prob', 'y_test', 'test_prob']

Merged prediction saved: C:\Users\zhd\LungCancer\03.Ensemble learning\02_Boosting\all_predictions.csv
Evaluation table with bootstrap CI saved: C:\Users\zhd\LungCancer\03.Ensemble learning\02_Boosting\all_evaluation_metrics.csv
All figures & tables saved to C:\Users\zhd\LungCancer\03.Ensemble learning\02_Boosting
